# VRSBench Image Captioning — PaliGemma 2 (3B, 448px) + LoRA + Length-Penalty Loss
Same architecture and overall pipeline as the SmolVLM2 run: LoRA finetune on the **full** VRSBench
training set, a length-penalty loss term pulling generated captions toward the dataset's median
length (53 words), a 9hr wall-clock training cap, and the same metric suite / 50-sample official
val eval at the end.

Unlike moondream2, PaliGemma **is** a standard `Trainer`-compatible HF model
(`PaliGemmaForConditionalGeneration` + `AutoProcessor`) — `google/paligemma2-3b-pt-448`, the
pre-trained (not task-fine-tuned) 3B/448px checkpoint, chosen specifically because Google designed
the `-pt-` checkpoints for exactly this kind of downstream LoRA fine-tune (per the PaliGemma 2
release blog and Google's own DOCCI long-captioning fine-tunes of the same checkpoint family). So
Section 9 goes back to a `Trainer` subclass with a custom `compute_loss`, same as the SmolVLM
notebook — no manual training loop needed here. The fine-tuning approach below (prefix/suffix
tokenization via `processor(..., suffix=...)`, LoRA on the Gemma-2 language model with the vision
tower + projector frozen, `paged_adamw`-style 8-bit optimizer) follows Hugging Face's own
`Fine_tune_PaliGemma.ipynb` and `paligemma.py` (`merveenoyan/smol-vision` — the reference notebook
linked from the official PaliGemma 2 launch blog), cross-checked against the PaliGemma 2 model card
and `transformers` docs for the `PaliGemmaForConditionalGeneration`/`PaliGemmaProcessor` API and the
`"<image>caption {lang}"` prompt convention.

## 1. Setup — installs, imports, data + model download (everything up front)

In [1]:
!pip install -q "transformers>=4.47" accelerate peft bitsandbytes trl num2words
!pip install -q sentence-transformers nltk rouge-score pycocoevalcap

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 17.9 MB/s eta 0:00:00


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import json
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

RUN_START = time.time()
def elapsed_min():
    return (time.time() - RUN_START) / 60

Device: cuda


In [3]:
import torch

print("Visible GPUs:", torch.cuda.device_count())
print("GPU:", torch.cuda.get_device_name(0))

Visible GPUs: 1
GPU: Tesla T4


In [4]:
# --- Dataset paths (mounted Kaggle input, no download needed) ---
DATA_DIR = "/kaggle/input/datasets/ayaanmustafa/vrs-bench/vrsbench_data"
ANNOT_FILE = os.path.join(DATA_DIR, "VRSBench_train_caption.json")
IMG_DIR = os.path.join(DATA_DIR, "Images_train", "Images_train")
VAL_IMG_DIR = os.path.join(DATA_DIR, "Images_val", "Images_val")
EVAL_CAP_JSON = os.path.join(DATA_DIR, "VRSBench_EVAL_Cap.json")

# --- Output directory (Kaggle persists /kaggle/working/ as notebook output) ---
OUTPUT_DIR = "/kaggle/working/paligemma2_vrsbench_final"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

login(token=HF_TOKEN)

print("Hugging Face authentication successful")

Hugging Face authentication successful


In [6]:
# --- Model + processor download (pulled once here, used by every later cell) ---
from transformers import PaliGemmaForConditionalGeneration, PaliGemmaProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

MODEL_NAME = "google/paligemma2-3b-pt-448"   # "pt" = pre-trained, designed for downstream fine-tuning (per Google's own release notes)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

processor = PaliGemmaProcessor.from_pretrained(MODEL_NAME)
model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
)

# Freeze the SigLIP vision tower and the vision->language projector — only the Gemma-2 language
# model gets LoRA (matching the SmolVLM run, where the vision encoder also stays frozen).
for p in model.model.vision_tower.parameters():
    p.requires_grad = False

for p in model.model.multi_modal_projector.parameters():
    p.requires_grad = False

print(f"Model + processor downloaded. Elapsed: {elapsed_min():.1f} min")

preprocessor_config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/34.6M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Model + processor downloaded. Elapsed: 1.2 min


## 2. Build DataFrame from train captions

In [7]:
DATA_DIR = "/kaggle/input/datasets/ayaanmustafa/vrs-bench/vrsbench_data"
ANNOT_FILE = os.path.join(DATA_DIR, "VRSBench_train.json")  # update filename if different
IMG_DIR = os.path.join(DATA_DIR, "Images_train", "Images_train")

with open(ANNOT_FILE, "r") as f:
    raw = json.load(f)

caption_rows = []

for item in raw:
    conversations = item["conversations"]

    if "[caption]" in conversations[0]["value"].lower():
        caption_rows.append({
            "id": item["id"],
            "image": item["image"],
            "caption": conversations[1]["value"]
        })

df = pd.DataFrame(caption_rows)
# Normalize columns: expect a filename column and a caption column
# (adjust the renames below to match the printed column names above)
rename_map = {}
for c in df.columns:
    lc = c.lower()
    if "image" in lc and "path" not in lc:
        rename_map[c] = "filename"
    elif lc in ("image_path", "path"):
        rename_map[c] = "filename"
    elif "caption" in lc:
        rename_map[c] = "caption"

df = df.rename(columns=rename_map)
df["image_path"] = df["filename"].apply(lambda x: os.path.join(IMG_DIR, os.path.basename(str(x))))

df = df[["image_path", "caption"]].dropna().reset_index(drop=True)
print(df.shape)
df.head()

(20264, 2)


,image_path,caption
0,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, shows a r..."
1,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,This aerial image from Google Earth shows a de...
2,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, showcases..."
3,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, shows a f..."
4,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, showcases..."


## 3. Train set — full dataset

In [8]:
from sklearn.model_selection import train_test_split

train_df, _ = train_test_split(df, test_size=0.1, random_state=SEED)

# Full dataset — no subsampling. Training is wall-clock capped (Section 9), so the full pool
# just means more of it gets seen; the time callback still governs when training stops.
train_df = train_df.reset_index(drop=True)

print("Train:", len(train_df))

Train: 18237


## 4. LoRA setup
Settings now match the **official** reference fine-tuning script (`merveenoyan/smol-vision/paligemma.py`,
linked from the PaliGemma 2 launch blog) rather than the SmolVLM run's own numbers: rank **8** (not 16),
alpha left at PEFT's default (**8**, i.e. a 1x scale — the official script doesn't set `lora_alpha`
explicitly), and no LoRA dropout (the official script doesn't set one either). Target modules are
unchanged — PaliGemma's Gemma-2 language model uses the same Llama-style naming as SmolVLM.

In [9]:
lora_config = LoraConfig(
    r=8,                # matches the official smol-vision paligemma.py reference script's LoRA rank
    lora_alpha=8,        # alpha == r (script leaves this at peft's default, i.e. scale factor 1)
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 11,876,352 || all params: 3,045,003,504 || trainable%: 0.3900


## 5. Metric-aware prompt
Median caption length in VRSBench is 53 words — stated explicitly, same as the SmolVLM run.
PaliGemma's `-pt-` checkpoints are pre-trained on short, fixed task-prefix conventions (`"<image>caption
en"`, `"<image>answer en {question}"`, etc. — see the official inference snippet), not on long
system-style instructions with embedded few-shot images the way SmolVLM's chat template allows. So
rather than fighting that convention with a long prefix, the task prefix stays close to the
pre-training format and the explicit length target is left to the length-penalty loss (Section 8) to
teach through gradient signal, same division of labor as the SmolVLM run intended (prompt states the
target, the loss enforces it).

In [10]:
MEDIAN_WORDS = 53
PROMPT_PREFIX = f"<image>caption en, about {MEDIAN_WORDS} words"

## 6. PyTorch Dataset
Same loose safety cap (90 words) as the SmolVLM run — not the primary length-control mechanism,
just a guard against one outlier caption dominating a batch. The length-penalty loss (Section 8)
does the actual length-shaping via gradient signal.

In [11]:
from torch.utils.data import Dataset

class VRSBenchPaliGemmaDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        caption = row["caption"]
        words = caption.split()
        if len(words) > 90:   # loose safety cap only, not the primary length-control mechanism
            caption = " ".join(words[:90])
        return {"image": image, "prefix": PROMPT_PREFIX, "caption": caption}


train_dataset = VRSBenchPaliGemmaDataset(train_df)
print(len(train_dataset))

18237


## 7. Collator
Uses `PaliGemmaProcessor`'s built-in `suffix=` argument (the convention from Google/HF's own
fine-tuning notebook): the processor builds `[prefix tokens][BOS-anchored image tokens][suffix
tokens]`, gives the prefix + image span full (non-causal) attention, and automatically masks the
prefix and padding positions to `-100` in `labels` so only the caption (`suffix`) tokens are
trained on — no manual image-token masking needed, unlike SmolVLM's collator.

In [12]:
def collate_fn(examples):
    images = [ex["image"] for ex in examples]
    prefixes = [ex["prefix"] for ex in examples]
    suffixes = [ex["caption"] for ex in examples]

    batch = processor(
        text=prefixes,
        images=images,
        suffix=suffixes,
        return_tensors="pt",
        padding="longest",
    )
    return batch

## 8. Length-penalty loss — penalizes deviation from the median (53 words)
Same structure as the SmolVLM run: a `Trainer` subclass overriding `compute_loss`, adding an
explicit auxiliary term on top of the model's own cross-entropy loss. For each sample, count the
non-masked label tokens (the caption/`suffix` tokens actually being trained on, since the processor
already masked prefix + image + padding positions to `-100`) and penalize the squared deviation from
a target token count corresponding to `MEDIAN_WORDS`.

In [13]:
from transformers import Trainer

# Rough token-per-word ratio for this tokenizer, used to convert the 53-word target into a token-count target
SAMPLE_TEXT = " ".join(["word"] * MEDIAN_WORDS)
TARGET_TOKEN_COUNT = len(processor.tokenizer(SAMPLE_TEXT, add_special_tokens=False)["input_ids"])
print("Target token count for median length:", TARGET_TOKEN_COUNT)

LENGTH_PENALTY_LAMBDA = 0.05   # weight of the length term relative to CE loss — a firmer pull than before

class LengthPenalizedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        ce_loss = outputs.loss

        labels = inputs["labels"]
        response_lengths = (labels != -100).sum(dim=-1).float()  # caption tokens actually trained on, per sample

        deviation = (response_lengths - TARGET_TOKEN_COUNT) ** 2
        length_loss = deviation.mean() / (TARGET_TOKEN_COUNT ** 2)   # normalized so it stays a small, stable term

        loss = ce_loss + LENGTH_PENALTY_LAMBDA * length_loss

        if self.state.global_step % 20 == 0:
            print(f"  [step {self.state.global_step}] ce={ce_loss.item():.4f} "
                  f"len_penalty={length_loss.item():.4f} avg_response_tokens={response_lengths.mean().item():.1f}")

        return (loss, outputs) if return_outputs else loss

Target token count for median length: 53


In [14]:
from sentence_transformers import SentenceTransformer

bert_model = SentenceTransformer("all-MiniLM-L6-v2", device=str(DEVICE))

def get_ngrams(tokens, n):
    return [" ".join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

def bert_bleu4(candidate, reference, alpha=0.5):
    c_tokens = candidate.lower().split()
    r_tokens = reference.lower().split()
    c_ngrams_by_n = [get_ngrams(c_tokens, n) for n in range(1, 5)]
    r_ngrams_by_n = [get_ngrams(r_tokens, n) for n in range(1, 5)]
    all_texts = [g for grams in c_ngrams_by_n + r_ngrams_by_n for g in grams]
    if not all_texts:
        return 0.0
    embs = bert_model.encode(all_texts, convert_to_tensor=True, show_progress_bar=False, batch_size=256)
    embs = F.normalize(embs, dim=-1)
    idx = 0
    c_embs_by_n, r_embs_by_n = [], []
    for grams in c_ngrams_by_n:
        c_embs_by_n.append(embs[idx: idx + len(grams)]); idx += len(grams)
    for grams in r_ngrams_by_n:
        r_embs_by_n.append(embs[idx: idx + len(grams)]); idx += len(grams)
    log_p_sum = 0.0
    for n in range(4):
        C_emb, R_emb = c_embs_by_n[n], r_embs_by_n[n]
        if len(C_emb) == 0 or len(R_emb) == 0:
            p_n = 1e-9
        else:
            sims = torch.mm(R_emb, C_emb.T)
            p_n = max(sims.max(dim=1).values.mean().item(), 1e-9)
        log_p_sum += np.log(p_n)
    L_c, L_r = len(c_tokens), max(len(r_tokens), 1)
    lp = np.exp(-alpha * abs(L_c - L_r) / L_r)
    return lp * np.exp(log_p_sum / 4)

def corpus_bert_bleu4(candidates, references):
    return float(np.mean([bert_bleu4(c, r) for c, r in zip(candidates, references)]))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt", quiet=True)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from pycocoevalcap.cider.cider import Cider

smoothie = SmoothingFunction().method4
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def compute_bleu_n(candidates, references, n):
    weights = tuple([1.0 / n] * n + [0.0] * (4 - n))
    scores = []
    for c, r in zip(candidates, references):
        c_tok, r_tok = c.lower().split(), [r.lower().split()]
        scores.append(sentence_bleu(r_tok, c_tok, weights=weights, smoothing_function=smoothie))
    return float(np.mean(scores))

def compute_meteor(candidates, references):
    return float(np.mean([meteor_score([r.lower().split()], c.lower().split()) for c, r in zip(candidates, references)]))

def compute_rouge_l(candidates, references):
    return float(np.mean([rouge.score(r, c)["rougeL"].fmeasure for c, r in zip(candidates, references)]))

def compute_cider(candidates, references):
    gts = {i: [r] for i, r in enumerate(references)}
    res = {i: [c] for i, c in enumerate(candidates)}
    score, _ = Cider().compute_score(gts, res)
    return float(score)

def compute_avg_len(candidates):
    return float(np.mean([len(c.split()) for c in candidates]))

def compute_all_metrics(candidates, references):
    return {
        "BLEU-1": compute_bleu_n(candidates, references, 1),
        "BLEU-2": compute_bleu_n(candidates, references, 2),
        "BLEU-3": compute_bleu_n(candidates, references, 3),
        "BLEU-4": compute_bleu_n(candidates, references, 4),
        "METEOR": compute_meteor(candidates, references),
        "ROUGE_L": compute_rouge_l(candidates, references),
        "CIDEr": compute_cider(candidates, references),
        "Avg_L": compute_avg_len(candidates),
        "BERT-BLEU4": corpus_bert_bleu4(candidates, references),
    }

## 9. Training — wall-clock capped to 9hr
Optimizer settings (`adamw_hf`, `weight_decay=1e-6`, `adam_beta2=0.999`, `learning_rate=2e-5`) now
match the official reference script exactly, rather than the 8-bit-optimizer convention used for the
SmolVLM/moondream2 runs. `num_train_epochs` is still set high as an upper bound with a 9hr wall-clock
callback stopping training — the reference script instead uses a fixed `num_train_epochs=3` over its
much smaller (~3k-sample) VQAv2 slice, which doesn't fit this run's full-dataset, time-capped setup.

In [16]:
from transformers import TrainingArguments, TrainerCallback

TARGET_TRAIN_MINUTES = 540   # 9hr wall-clock cap on training

class TimeLimitCallback(TrainerCallback):
    def __init__(self, target_minutes):
        self.target_seconds = target_minutes * 60
        self.start_time = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self.start_time
        if elapsed >= self.target_seconds:
            print(f"\n[TimeLimitCallback] Reached {elapsed/60:.1f} min — stopping training gracefully "
                  f"(completed {state.global_step} steps, epoch {state.epoch:.2f}).")
            control.should_training_stop = True
        return control

use_bf16 = torch.cuda.is_bf16_supported()

training_args = TrainingArguments(
    output_dir="/kaggle/working/paligemma2_vrsbench_checkpoints",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=16,

    learning_rate=2e-4,
    weight_decay=0.01,

    num_train_epochs=50,

    logging_steps=20,
    save_steps=100,
    save_total_limit=2,

    bf16=use_bf16,
    fp16=not use_bf16,

    gradient_checkpointing=True,

    optim="adamw_torch",

    report_to="none",
    remove_unused_columns=False,
)

trainer = LengthPenalizedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
    callbacks=[TimeLimitCallback(TARGET_TRAIN_MINUTES)],
)

train_result = trainer.train()
print(train_result)
print(f"Training took {elapsed_min():.1f} min total (incl. setup)")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


  [step 0] ce=5.1490 len_penalty=0.1424 avg_response_tokens=33.0
  [step 0] ce=4.5058 len_penalty=0.0174 avg_response_tokens=46.0
  [step 0] ce=3.9501 len_penalty=0.0228 avg_response_tokens=61.0
  [step 0] ce=3.4162 len_penalty=0.0128 avg_response_tokens=47.0
  [step 0] ce=3.5649 len_penalty=0.1029 avg_response_tokens=70.0
  [step 0] ce=4.2073 len_penalty=0.0911 avg_response_tokens=37.0
  [step 0] ce=3.6125 len_penalty=0.1285 avg_response_tokens=34.0
  [step 0] ce=4.2795 len_penalty=0.0057 avg_response_tokens=49.0
  [step 0] ce=3.6045 len_penalty=0.0014 avg_response_tokens=51.0
  [step 0] ce=3.7489 len_penalty=0.9260 avg_response_tokens=104.0
  [step 0] ce=4.0098 len_penalty=0.0057 avg_response_tokens=57.0
  [step 0] ce=4.1758 len_penalty=0.0014 avg_response_tokens=55.0
  [step 0] ce=4.0965 len_penalty=0.2994 avg_response_tokens=82.0
  [step 0] ce=4.0542 len_penalty=0.0174 avg_response_tokens=46.0
  [step 0] ce=3.6196 len_penalty=0.8900 avg_response_tokens=103.0
  [step 0] ce=4.0385 le

Step,Training Loss
20,39.673193
40,29.733591
60,27.927075
80,27.053387
100,26.545782
120,26.034052
140,26.138293
160,25.451314
180,25.436620


  [step 20] ce=2.3718 len_penalty=0.0014 avg_response_tokens=55.0
  [step 20] ce=1.4383 len_penalty=0.0356 avg_response_tokens=63.0
  [step 20] ce=1.9090 len_penalty=0.6892 avg_response_tokens=97.0
  [step 20] ce=2.1097 len_penalty=0.1723 avg_response_tokens=75.0
  [step 20] ce=2.4184 len_penalty=0.0089 avg_response_tokens=58.0
  [step 20] ce=2.5798 len_penalty=0.0000 avg_response_tokens=53.0
  [step 20] ce=2.3504 len_penalty=0.1570 avg_response_tokens=32.0
  [step 20] ce=1.8628 len_penalty=0.0089 avg_response_tokens=58.0
  [step 20] ce=1.9598 len_penalty=0.2225 avg_response_tokens=78.0
  [step 20] ce=2.0755 len_penalty=0.3204 avg_response_tokens=83.0
  [step 20] ce=1.4009 len_penalty=0.0356 avg_response_tokens=43.0
  [step 20] ce=1.6686 len_penalty=0.5984 avg_response_tokens=94.0
  [step 20] ce=1.7709 len_penalty=0.1153 avg_response_tokens=71.0
  [step 20] ce=1.8829 len_penalty=0.0288 avg_response_tokens=62.0
  [step 20] ce=1.5652 len_penalty=0.2595 avg_response_tokens=26.0
  [step 20

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


  [step 100] ce=1.8121 len_penalty=0.2595 avg_response_tokens=80.0
  [step 100] ce=2.2450 len_penalty=0.0174 avg_response_tokens=60.0
  [step 100] ce=1.2809 len_penalty=0.1153 avg_response_tokens=71.0
  [step 100] ce=1.6778 len_penalty=0.0356 avg_response_tokens=63.0
  [step 100] ce=1.6484 len_penalty=0.1723 avg_response_tokens=75.0
  [step 100] ce=1.7569 len_penalty=0.0356 avg_response_tokens=43.0
  [step 100] ce=1.7513 len_penalty=0.1723 avg_response_tokens=75.0
  [step 100] ce=1.9799 len_penalty=0.3204 avg_response_tokens=83.0
  [step 100] ce=1.9352 len_penalty=0.1570 avg_response_tokens=74.0
  [step 100] ce=1.6977 len_penalty=0.0014 avg_response_tokens=51.0
  [step 100] ce=1.7576 len_penalty=0.0057 avg_response_tokens=57.0
  [step 100] ce=1.6425 len_penalty=0.0089 avg_response_tokens=48.0
  [step 100] ce=1.4205 len_penalty=0.2595 avg_response_tokens=80.0
  [step 100] ce=1.4563 len_penalty=0.0089 avg_response_tokens=58.0
  [step 100] ce=2.1375 len_penalty=0.0911 avg_response_tokens=

## 10. Save final model to the output directory

In [17]:
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("Saved final model + processor to:", OUTPUT_DIR)
!ls {OUTPUT_DIR}

Saved final model + processor to: /kaggle/working/paligemma2_vrsbench_final
adapter_config.json	   processor_config.json  tokenizer_config.json
adapter_model.safetensors  README.md		  tokenizer.json


## 11. Metric suite

## 12. Load official val split, 50-sample eval

In [18]:
with open(EVAL_CAP_JSON, "r") as f:
    eval_raw = json.load(f)

eval_df = pd.DataFrame(eval_raw)
if "type" in eval_df.columns:
    eval_df = eval_df[eval_df["type"] == "caption"].reset_index(drop=True)

eval_rename = {}
for c in eval_df.columns:
    lc = c.lower()
    if "image" in lc and "path" not in lc:
        eval_rename[c] = "filename"
    elif lc in ("image_path", "path"):
        eval_rename[c] = "filename"
    elif "caption" in lc or lc in ("ground_truth", "answer"):
        eval_rename[c] = "caption"

eval_df = eval_df.rename(columns=eval_rename)
eval_df["image_path"] = eval_df["filename"].apply(lambda x: os.path.join(VAL_IMG_DIR, os.path.basename(str(x))))
eval_df = eval_df[["image_path", "caption"]].dropna().reset_index(drop=True)

EVAL_SUBSET_SIZE = 50
eval_subset_df = eval_df.sample(n=min(EVAL_SUBSET_SIZE, len(eval_df)), random_state=SEED).reset_index(drop=True)
print("Eval samples used:", len(eval_subset_df))

Eval samples used: 50


## 13. Inference
Follows PaliGemma's standard inference pattern (`processor(prompt, image, return_tensors="pt")` →
`model.generate` → decode only the newly generated tokens past the input length) — no hard
word-count truncation, same rationale as the SmolVLM run: seeing what the length-penalty loss
learned on its own.

In [19]:
model.eval()

@torch.no_grad()
def generate_caption(image, max_new_tokens=120):
    inputs = processor(text=PROMPT_PREFIX, images=image, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
    )
    generated = output_ids[:, input_len:]
    text_out = processor.batch_decode(generated, skip_special_tokens=True)[0]
    return text_out.strip()


def validate(eval_df):
    preds, refs = [], []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="val"):
        image = Image.open(row["image_path"]).convert("RGB")
        pred = generate_caption(image)
        preds.append(pred)
        refs.append(row["caption"])
    metrics = compute_all_metrics(preds, refs)
    return metrics, preds, refs


metrics, preds, refs = validate(eval_subset_df)
print(metrics)
print(f"Avg predicted length: {np.mean([len(p.split()) for p in preds]):.1f} words (target median: {MEDIAN_WORDS})")
print(f"Total run time: {elapsed_min():.1f} min")

val:   0%|          | 0/50 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


{'BLEU-1': 0.32027024389539216, 'BLEU-2': 0.18041718222511785, 'BLEU-3': 0.11131476356177686, 'BLEU-4': 0.07670214853299852, 'METEOR': 0.2822677667680277, 'ROUGE_L': 0.3326990470640914, 'CIDEr': 0.20722427625447018, 'Avg_L': 41.48, 'BERT-BLEU4': 0.5682876624098243}
Avg predicted length: 41.5 words (target median: 53)
Total run time: 550.6 min


## 14. Sample predictions

In [20]:
for p, r in list(zip(preds, refs))[:5]:
    print(f"PRED ({len(p.split())}w):", p)
    print(f"REF  ({len(r.split())}w):", r)
    print("-" * 60)

PRED (50w): The image from GoogleEarth features a large ship docked at an industrial facility. The vessel is positioned in the center of the frame with its length extending towards the bottom-right corner and it appears to be surrounded by structures that appear like docks or piers adjacent to the water's edge.
REF  (24w): The image from GoogleEarth shows a docking area with a ship. The ship has a white and green color scheme and is very long.
------------------------------------------------------------
PRED (36w): The aerial image from GoogleEarth features a recreational area with two basketball courts. The right-most court is located towards the bottom of the frame and has clear markings for playing games such as hoops or shots.
REF  (35w): This aerial image from GoogleEarth features a variety of elements including a prominent basketball court. The basketball court is prominently placed in the lower central part of the image, paved in gray with white markings.
-----------------------